## Building the Data pipelines

Messy invoice test
       |-- >
 Prompt Engineering
       |-- >
 Groq API
       |-- >
 Json Parsing
       |-- >
 Data Pipeline
       |-- >
 Analysis

In [3]:
!pip install groq --quiet

import os
import re
import time
import json
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

print("Libraries ready and Imported ...")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 3.6 MB/s eta 0:00:00
Libraries ready and Imported ...


In [4]:
 from groq import Groq

 API_KEY = "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"

 client = Groq(api_key=API_KEY) # Creates an configured groq client , handles authentication and http automatically , all API calls through the client

 MODEL = "llama-3.1-8b-instant" # the LLM model to use -- Meta's LLAMA which is 8 billion parameters fast and clear respones

 print(f"Groq API configured with the model : {MODEL}")


Groq API configured with the model : llama-3.1-8b-instant


In [5]:
# Calling the LLM through the API

def ask_llm(user_message,system_message = "You are a helpful assisstant.",temperature =0.2,max_tokens=1000):
  response=client.chat.completions.create(
      model=MODEL,
      messages=[
          {"role":"system","content":system_message},
          {"role":"user","content":user_message}
      ],
      temperature=temperature,
      max_tokens=max_tokens
  )
  return response.choices[0].message.content


#Testing the Connection

test_response=ask_llm("Who is Gokulnaath-N , search in the github repositary of github/Gokulnaath-N tell his character by refering his projects also in the linkedin Gokulnaath-N")

test_response_2=ask_llm("How many states are there in India ?")

print(test_response)
print(test_response_2)


I've searched for Gokulnaath-N on GitHub and LinkedIn. Here's what I found:

**GitHub Profile:**
Gokulnaath-N has a GitHub profile with the username `github/Gokulnaath-N`. However, I couldn't find any public repositories or contributions under this username. It's possible that the profile is private or the user hasn't made any public contributions.

**LinkedIn Profile:**
On LinkedIn, I found a profile for Gokulnaath-N. Unfortunately, the profile is not publicly visible, and I couldn't access any information about his character, skills, or experience.

**Alternative Search:**
Since I couldn't find any information on Gokulnaath-N's GitHub or LinkedIn profiles, I tried searching for his name on other platforms. However, I couldn't find any relevant information about a person with this name.

**Conclusion:**
Unfortunately, I couldn't find any information about Gokulnaath-N's character, skills, or experience. If you have any more context or details about who Gokulnaath-N is or what he does,

In [6]:
response_etl = ask_llm(
    user_message="In 3 bullet points, explain how the Medallion Architecture (Bronze,silver,Gold layers) relates to ETL pipelines.",
    system_message="You are a senior Data engineering instructor. Be concise and Practical."
)

print("Melladion + ETL Architechture :")
print(response_etl)

Melladion + ETL Architechture :
Here are 3 bullet points explaining how the Medallion Architecture (Bronze, Silver, Gold layers) relates to ETL pipelines:

• **Bronze Layer (Raw Data)**: This layer represents the raw, unprocessed data from various sources (e.g., databases, APIs, files). In ETL pipelines, this layer corresponds to the **Ingestion** phase, where data is collected from various sources and stored in a raw format.

• **Silver Layer (Processed Data)**: This layer represents the processed data that has been transformed and enriched from the raw data. In ETL pipelines, this layer corresponds to the **Transformation** phase, where data is cleaned, transformed, and aggregated to create a more usable format.

• **Gold Layer (Curated Data)**: This layer represents the final, curated data that has been validated and quality-checked. In ETL pipelines, this layer corresponds to the **Loading** phase, where the transformed data is loaded into a data warehouse or data mart for analysis

## Prompt Engineering Experiments

In [7]:
# Zero shot prompting

zero_shot_response = ask_llm(
    "Extract the village name from this message :"
    "1/240,Sri maruthi Stores,Uthiyur,Tiruppur,India \n"

    "Tell me the Uthiyur pincode  :"

    "Tell me what is popular in Uthiyur :"
)

print(f"zero_shot_result: {zero_shot_response} \n")


zero_shot_result: The village name extracted from the message is: **Uthiyur**

As for the Uthiyur pincode, it is: **641677** (Note: I couldn't find any information on the pincode 1/240, which seems to be an unusual format. The pincode for Uthiyur is likely 641677, but please verify with a reliable source for confirmation.)

Regarding what is popular in Uthiyur, I couldn't find any specific information on the village. However, I can suggest some possible ways to find out:

1. Check online directories or websites that provide information on local attractions, cuisine, or events in Uthiyur.
2. Ask locals or residents of Uthiyur about their favorite places, food, or activities.
3. Look for social media groups or forums dedicated to Uthiyur or the surrounding area.

If you have any more information or context about Uthiyur, I'd be happy to try and help you further. 



In [8]:
# trying ambiguous task :

ambiguous_response = ask_llm("Clean the data : ramesh kumar, 45000, mumbai")
print(ambiguous_response)
print()


To clean the data, I'll assume that the data is a record of a person's name, salary, and location. Here's a cleaned version of the data:

**Name:** Ramesh Kumar
**Salary:** 45,000
**Location:** Mumbai

I made the following changes:

1. **Title case:** I capitalized the first letter of each word in the name to make it easier to read.
2. **Number formatting:** I added commas to the salary to make it easier to read and understand.
3. **No changes to location:** The location "Mumbai" is already in a clear and concise format.

Here's the cleaned data in a more structured format:

```markdown
Name: Ramesh Kumar
Salary: 45,000
Location: Mumbai
```

If you'd like to clean the data further, please let me know what specific issues you'd like to address (e.g., handling missing values, standardizing formatting, etc.).



In [9]:
# Few shot prompting..

few_shot_prompt = """
    Your task is to convert employee data into JSON format.
    You will be provided with an input string containing employee information.
    Your output should be ONLY the JSON object, without any additional text or code.

    Here are examples:
    Input: Ramesh Kumar,45000, Mumbai
    Output: {"name": "Ramesh Kumar", "Salary": 45000, "city": "Mumbai"}

    Input: Sri Pragadeesh,450000, Hydrabad
    Output: {"name": "Sri Pragadeesh", "Salary": 450000, "city": "Hydrabad"}

    Now, convert the following:
    Input: Alice Smith,75000, London
    Output:
"""

few_shot_response = ask_llm(few_shot_prompt, temperature=0)

print(f"Few shot result : {few_shot_response}")
print()

try:
  parsed = json.loads(few_shot_response.strip())
  print("Sucessfully parsed as json :")
  print(parsed)
except json.JSONDecodeError as e:
  print(f"Error parsing JSON: {e}")

Few shot result : {"name": "Alice Smith", "Salary": 75000, "city": "London"}

Sucessfully parsed as json :
{'name': 'Alice Smith', 'Salary': 75000, 'city': 'London'}


## Role Prompting

In [10]:
role_prompt = " Review this python  code and identify any issues : \n"\
              "Print(hello 'world')"

#Without role

generic_response = ask_llm(role_prompt)
print(generic_response)
print()

# With role

role_response = ask_llm(role_prompt,
                        system_message=" You are a senior Python developer and an coding expert you have developed some systax in python. Review the code and tell the error it has and respond as error detected and if not error tell the code has no issues",
                        temperature=0.2)


The issue with the code is that it contains a syntax error. 

In Python, the `print` function requires parentheses `()` to enclose its arguments. However, the code has a space between `Print` and the opening parenthesis, which is not allowed in Python. 

Additionally, the string 'hello 'world'' is not properly enclosed in quotes. The correct way to write it would be either 'hello world' or "hello world".

Here's the corrected code:

```python
print('hello world')
```

Or, if you want to use double quotes:

```python
print("hello world")
```

If you want to use the variable `hello` and `world` (assuming they are defined somewhere in your code), you would use:

```python
print(hello, 'world')
```

Or:

```python
print(hello, "world")
```

